# v104_two_stage — v101 as stage 1, learned candidate filter, stage 2 trained at test density

| Field | Value |
|---|---|
| **Version** | `v104_two_stage` |
| **Plan group** | C5 + D3 (competition features, matcher), A5 (candidate filter) |
| **Parent version** | v103 (v101 matcher, rule tuned on the mock) |
| **Author** | M1 rajaguru2004 |
| **Date** | 2026-09-26 |
| **Status** | shortlisted |

v103 showed how v101 behaves at the test's decoy density (mock fold). v104 keeps v101 as a
**stage-1** scorer and adds a **stage-2** matcher trained inside the mock fold, where every
entity meets as many same-name records and 1-to-1 rivals as on test:

```
mock fold (test shape) ─► v101 blocking (cached) ─► v101 features + p1 on every pair
    ├─► filter: p1 >= floor and top max_cands per S1  ─► final candidate set (smaller file)
    ├─► competition features from p1 over ALL pairs (rank / best rival, S1 side + pool side)
    └─► anchor features: each kept candidate vs its entity's best OTHER candidate
kept pairs of the mock's fit entities ─► stage-2 GBDT, cross-fitted in 2 parts
stage-2 probs ─► 1-to-1 across every present S1 ─► rule tuned on mock tune S1 ─► mock val F0.5
```

## 1. Hypothesis

* **Change vs parent (v103):** a second matcher on top of v101's probabilities. Its inputs
  are v101's 53 pair features, twelve competition features (`stacking.STACK_COLUMNS`: the
  pair's rank, best rival and gap among its S1 entity's candidates and among the S1
  entities holding the same pool record, likely-match counts and probability sums on both
  sides, the pool record's degree) and five anchor features (`stacking.ANCHOR_COLUMNS`:
  the candidate compared with its entity's best other candidate, since the ~3.5 true
  records of one business resemble each other and a same-name decoy does not). It is trained on the mock fold's `fit` entities (none of which v101 saw), so it
  learns what a probability means when decoys are as dense as on test. The rule is re-tuned
  on the mock tune entities. The candidate set shrinks to the pairs stage 2 scores.
* **Why it should raise mock F0.5:** at test density the typical false merge is a
  same-name record that v101 scores high on its own. Whether a better record exists for the
  same entity, a better entity exists for the same record, or the record's address departs
  from the entity's other records, is exactly what v101 cannot see and stage 2 can. Two
  ablations (§5) attribute the gain: without anchors, and with v101's features only (a
  density-trained single stage).
* **Expected effect:** fewer false merges and singleton merges at equal recall; candidates
  per S1 from ~35 to ~10 with ≤ 0.002 candidate-recall loss.
* **Discard if:** mock F0.5 does not beat v103 by more than 0.002.

## 2. Setup

v101's configuration is stage 1 (its artifacts are loaded, nothing in it is retrained).
`TwoStageConfig` holds the filter (`floor`, `max_cands`), the cross-fitting (2 parts, seed
6161), the early-stopping sample (50k mock tune entities) and the stage-2 GBDT parameters.

In [1]:
import json
import shutil
import subprocess
import sys
import time
from dataclasses import asdict

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

from entity_resolution import config as C
from entity_resolution.data import isin
from entity_resolution.decision import decide
from entity_resolution.evaluate import blocking_report, error_samples
from entity_resolution.features import DEFAULT_GROUPS
from entity_resolution.mock import build_mock, target_shape
from entity_resolution.model import MatcherParams
from entity_resolution.pipeline import (
    Fitted, PipelineConfig, mock_scores, peak_rss_gb, tune_mock,
)
from entity_resolution.split import load_fold
from entity_resolution.tracking import log_result, timed
from entity_resolution.trainset import inner_split, sample_s1
from entity_resolution.twostage import (
    TwoStage, TwoStageConfig, fit_stage2, mock_scored, mock_stage1, run_test_two_stage,
)

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 50)
pd.set_option("display.max_columns", 30)

EXP_DIR = C.EXPERIMENTS / "v104_two_stage"
ARTIFACTS = EXP_DIR / "artifacts"
cfg = PipelineConfig(feature_groups=(*DEFAULT_GROUPS, "frequency"),
                     model=MatcherParams(n_estimators=4000))          # v101 = stage 1
stage1 = Fitted.load(C.EXPERIMENTS / "v101_name_frequency" / "artifacts", cfg)
tcfg = TwoStageConfig(floor=0.01, max_cands=16,
                      model=MatcherParams(backend="xgb", device="cuda",
                                          n_estimators=4000))
parent = json.loads((C.EXPERIMENTS / "v103_mock_rule" / "metrics.json").read_text())
# stage-1 outputs (kept pairs + stage-2 frame) are cached per stage-1 model, blocking and
# filter, so later stage-2 variants on the same stage 1 skip the ~15-min stage-1 passes
STAGE1_CACHE = (cfg.cache_dir / "stage1" /
                f"v101_{cfg.blocking.key()}_f{tcfg.floor}_k{tcfg.max_cands}_a{int(tcfg.anchors)}")
timings: dict[str, float] = {}
t_start = time.time()
print(json.dumps(tcfg.record(), indent=1))
print("parent v103 mock F0.5:", parent["mock_f05"], "rule:", parent["metrics"]["rule"])

{
 "floor": 0.01,
 "max_cands": 16,
 "folds": 2,
 "seed": 6161,
 "n_stop_s1": 50000,
 "anchors": true,
 "cohesion": false,
 "model": {
  "backend": "xgb",
  "num_leaves": 63,
  "learning_rate": 0.05,
  "n_estimators": 4000,
  "early_stopping": 100,
  "feature_fraction": 0.8,
  "bagging_fraction": 0.8,
  "bagging_freq": 1,
  "min_data_in_leaf": 200,
  "lambda_l2": 1.0,
  "max_bin": 255,
  "scale_pos_weight": 1.0,
  "seed": 42,
  "num_threads": 12,
  "device": "cuda",
  "min_child_weight": 1.0
 }
}
parent v103 mock F0.5: 0.9704 rule: {'tau_abs': 0.725, 'tau_rel': 0.7, 'tau_single': 0.775, 'max_matches': 11, 'one_to_one': True}


## 3. Data

The same mock fold as v103 (`build_mock`, seeds 5151 / 5152): per country the test's pool
size and pool-per-S1 ratio, v101's training sample dropped, every val and tune entity of a
kept cluster present. Its blocking is already cached.

In [2]:
with timed("load", timings):
    train = load_fold("train", columns=[C.COUNTRY])
    val = load_fold("val", columns=[C.COUNTRY])
    fit_fold, tune_fold = inner_split(train)
    fit_sample = sample_s1(fit_fold.s1, cfg.n_fit_s1)[C.ENTITY_ID]
    mock = build_mock(train, val, tune_fold.s1[C.ENTITY_ID], fit_sample, target_shape())
del train, val, fit_fold, tune_fold, fit_sample
display(mock.info)
mock.fold.summary()

,country,s1_train,pool_train,keep_frac,s1_kept,pool_kept,s1_present,pool_per_s1,test_pool_per_s1,present_val,present_tune,present_fit
0,India,883188,4133346,1.000,883188,4133346,709678,5.824,5.824,176522,176208,356948
1,US,1323633,6186873,0.617,815850,3816702,663049,5.756,5.756,163562,163325,336162


{'fold': 'mock',
 's1': 1372727,
 's2': 3878856,
 's3': 4071192,
 'true_pairs': 4753992,
 'singleton_share': 0.0558}

## 4. Method

### 4.1 Stage 1: v101 on every mock pair, filter, competition features

`mock_stage1` builds v101's features chunk by chunk, scores them with v101, keeps the pairs
with `p1 >= floor` among each entity's `max_cands` best, and adds the competition features
computed over **all** pairs of the country. The report compares candidate recall before
(all blocking candidates) and after the filter, per role: the filter must cost almost no
recall while cutting the candidate count.

In [3]:
t0 = time.time()
outs = mock_stage1(cfg, stage1, mock, tcfg, timings=timings, cache_dir=STAGE1_CACHE / "mock")
timings["stage1_seconds"] = round(time.time() - t0, 2)
kept = pd.concat([o.pairs for o in outs.values()], ignore_index=True)
print(f"stage 1 {timings['stage1_seconds']:.0f} s; pairs {sum(o.n_all for o in outs.values()):,}"
      f" -> kept {len(kept):,}")
rows = {}
for role in ("fit", "tune", "val"):
    part = mock.part(role)
    rows[role] = blocking_report(kept, part)
filter_report = pd.DataFrame(rows).T
filter_report[["pair_recall", "entity_recall", "ceiling_f_beta", "candidates_mean",
               "candidates_p95", "candidates_max"]]

stage 1 918 s; pairs 47,348,383 -> kept 6,311,331


,pair_recall,entity_recall,ceiling_f_beta,candidates_mean,candidates_p95,candidates_max
fit,0.964196,0.996389,0.987477,4.601084,9.0,16.0
tune,0.964089,0.996298,0.987441,4.595480,9.0,16.0
val,0.964386,0.996392,0.987585,4.592856,9.0,16.0


Candidate recall of the unfiltered blocking on the same roles, for comparison (the pairs are
read back from the mock's blocking cache files; nothing is recomputed).

In [4]:
cached = sorted((cfg.cache_dir / "pairs").glob(f"mock_*/*/pairs_{cfg.blocking.key()}.parquet"))
print([str(p.relative_to(cfg.cache_dir)) for p in cached])
all_pairs = pd.concat([pd.read_parquet(p, columns=[C.S1_ID, C.ENTITY_ID]) for p in cached],
                      ignore_index=True)
before = pd.DataFrame({r: blocking_report(all_pairs, mock.part(r)) for r in ("tune", "val")}).T
del all_pairs
before[["pair_recall", "entity_recall", "ceiling_f_beta", "candidates_mean", "candidates_p95"]]

['pairs/mock_663049_be158948_3816702_f184bbc7_mcd072aee/US/pairs_66540dae.parquet', 'pairs/mock_709678_fd324a23_4133346_476eb49c_mcd072aee/India/pairs_66540dae.parquet']


,pair_recall,entity_recall,ceiling_f_beta,candidates_mean,candidates_p95
tune,0.965179,0.996392,0.987799,34.483491,44.0
val,0.965493,0.996486,0.987942,34.498303,44.0


### 4.2 Stage 2: trained on the mock's fit entities, cross-fitted

Two models, each trained on half of the fit entities (by id hash) with early stopping on
the kept pairs of 50k tune entities. Fit entities are later scored by the model that did
not see them; tune and val entities by the mean of both. The importance table shows how
much of stage 2's evidence comes from the competition features.

In [5]:
t0 = time.time()
models, fit_info = fit_stage2(outs, mock, tcfg)
timings["fit_seconds"] = round(time.time() - t0, 2)
print(f"stage 2 fit {timings['fit_seconds']:.0f} s", json.dumps(fit_info, indent=1, default=str))
importance = pd.concat([m.importance() for m in models], axis=1).mean(axis=1)
importance.sort_values(ascending=False).head(25).rename("gain share").to_frame()

/home/suryaguru/StudioProjects/aws/business_entity_resolution/.venv/lib64/python3.12/site-packages/xgboost/core.py:774: UserWarning: [11:08:31] WARNING: /workspace/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


stage 2 fit 587 s {
 "fold0": {
  "rows": 1595991,
  "positive_rate": 0.7260373022153633,
  "best_iteration": 1813,
  "tune_logloss": 0.05745782234353881,
  "tune_auc": 0.996679456316735,
  "fit_seconds": 300.24
 },
 "fold1": {
  "rows": 1593066,
  "positive_rate": 0.7262800160194242,
  "best_iteration": 1645,
  "tune_logloss": 0.057386273579197275,
  "tune_auc": 0.9967033134268523,
  "fit_seconds": 275.76
 },
 "rows": 3189057,
 "positive_rate": 0.7261585478089605
}


,gain share
feature,
pool_gap,0.720287
p1,0.193426
s1_gap,0.013990
pool_p1_sum,0.005848
pool_best_other,0.005823
s1_p1_sum,0.004805
sim_name_addr_word,0.002433
nm_token_set,0.002373
fq_first_pool_l,0.002146


### 4.3 Decision rule on the mock tune entities

Stage-2 probabilities for every kept pair (fit entities out of fold), the pool-side 1-to-1
across all present entities of each country, then the standard grid on the tune entities.

In [6]:
t0 = time.time()
scored, report = mock_scored(outs, models, mock, tcfg)
scored.to_parquet(ARTIFACTS / "mock_scored.parquet", index=False)   # for decision-layer work
rule, table = tune_mock(scored, mock, cfg.grid)
timings["tune_seconds"] = round(time.time() - t0, 2)
print("rule:", rule)
table.sort_values("f_beta", ascending=False).head(8)

rule: DecisionRule(tau_abs=0.68, tau_rel=0.7, tau_single=0.68, max_matches=11, one_to_one=True)


,tau_abs,tau_rel,tau_single,max_matches,one_to_one,f_beta,n_pred,pair_precision,pair_recall,match_rate,stage
1067,0.46,0.7,0.51,11,True,0.974714,1109920,0.994929,0.939281,0.939676,grid
443,0.36,0.7,0.51,11,True,0.974712,1109929,0.994926,0.939285,0.939676,grid
1316,0.50,0.7,0.50,11,True,0.974709,1109937,0.994922,0.939289,0.939738,grid
68,0.30,0.7,0.50,11,True,0.974707,1109953,0.994914,0.939295,0.939738,grid
692,0.40,0.7,0.50,11,True,0.974706,1109948,0.994916,0.939292,0.939738,grid
446,0.36,0.7,0.56,11,True,0.974705,1109829,0.994970,0.939243,0.939405,grid
1070,0.46,0.7,0.56,11,True,0.974704,1109825,0.994971,0.939240,0.939405,grid
572,0.38,0.7,0.58,11,True,0.974703,1109783,0.994989,0.939221,0.939281,grid


## 5. Evaluation

Mock F0.5 on the val entities (all and per country) against the parent v103 (v101 with the
mock-tuned rule) and v101's own rule; public scores: v001 0.954, v101 0.955.

In [7]:
res = mock_scores(scored, mock, rule)
cols = ["f_beta", "f_beta_singletons", "f_beta_matched", "pair_precision", "pair_recall",
        "entities"]
display(res[cols])
cmp = pd.DataFrame({
    "v103 (parent)": {"mock_f05": float(parent["mock_f05"]),
                      **{f"mock_{c}": parent["metrics"]["calibration"]["v101|mock rule"][f"mock_{c}"]
                         for c in ("India", "US")}},
    "v104": {"mock_f05": res.loc["all", "f_beta"],
             **{f"mock_{c}": res.loc[c, "f_beta"] for c in ("India", "US")}},
}).T
cmp["delta_vs_parent"] = cmp["mock_f05"] - float(parent["mock_f05"])
cmp.round(4)

,f_beta,f_beta_singletons,f_beta_matched,pair_precision,pair_recall,entities
all,0.974439,0.986812,0.973702,0.994991,0.939033,340084.0
India,0.970044,0.985485,0.969125,0.994518,0.928576,176522.0
US,0.979182,0.988246,0.978643,0.995490,0.950337,163562.0


,mock_f05,mock_India,mock_US,delta_vs_parent
v103 (parent),0.9704,0.9651,0.9762,0.000
v104,0.9744,0.9700,0.9792,0.004


### Ablations

Stage 2 retrained (same parts, same early stopping) on two column subsets, each tuned on the
mock tune entities and scored on the mock val entities: without the anchor features, and
with v101's pair features only (what training at density alone buys).

In [8]:
from entity_resolution.stacking import ANCHOR_COLUMNS, STACK_COLUMNS
all_cols = list(next(iter(outs.values())).X.columns)
variants = {"no anchors": [c for c in all_cols if c not in ANCHOR_COLUMNS],
            "pair features only": [c for c in all_cols
                                   if c not in ANCHOR_COLUMNS and c not in STACK_COLUMNS]}
ablation = {"full": {"mock_f05": res.loc["all", "f_beta"],
                     "f_beta_singletons": res.loc["all", "f_beta_singletons"]}}
for name, cols in variants.items():
    t0 = time.time()
    m_ab, _ = fit_stage2(outs, mock, tcfg, columns=cols)
    s_ab, _ = mock_scored(outs, m_ab, mock, tcfg)
    r_ab, _ = tune_mock(s_ab, mock, cfg.grid)
    sc = mock_scores(s_ab, mock, r_ab)
    ablation[name] = {"mock_f05": sc.loc["all", "f_beta"],
                      "f_beta_singletons": sc.loc["all", "f_beta_singletons"]}
    print(name, f"{time.time() - t0:.0f} s", ablation[name])
    del m_ab, s_ab
ablation = pd.DataFrame(ablation).T
ablation["delta_vs_full"] = ablation["mock_f05"] - ablation.loc["full", "mock_f05"]
ablation.round(4)

no anchors 664 s {'mock_f05': np.float64(0.9742279710627345), 'f_beta_singletons': np.float64(0.9850855617771731)}


pair features only 899 s {'mock_f05': np.float64(0.9725999849608881), 'f_beta_singletons': np.float64(0.9784394787796327)}


,mock_f05,f_beta_singletons,delta_vs_full
full,0.9744,0.9868,0.0000
no anchors,0.9742,0.9851,-0.0002
pair features only,0.9726,0.9784,-0.0018


## 6. Error analysis

Error counts on the mock val entities, then samples of the remaining precision errors and
of the misses stage 2 still makes, raw records side by side.

In [9]:
def with_raw(sample: pd.DataFrame) -> pd.DataFrame:
    """Raw name / address of both sides from the Parquet cache (only the sampled ids)."""
    ids = pd.concat([sample[C.S1_ID], sample[C.ENTITY_ID]]).unique().tolist()
    raw = pd.concat([pq.read_table(C.DATASET / ".cache" / f"train_source{s}.parquet",
                                   columns=[C.ENTITY_ID, C.NAME, C.ADDRESS],
                                   filters=[(C.ENTITY_ID, "in", ids)]).to_pandas()
                     for s in C.SOURCES]).set_index(C.ENTITY_ID)
    return sample.assign(name_l=sample[C.S1_ID].map(raw[C.NAME]),
                         addr_l=sample[C.S1_ID].map(raw[C.ADDRESS]),
                         name_r=sample[C.ENTITY_ID].map(raw[C.NAME]),
                         addr_r=sample[C.ENTITY_ID].map(raw[C.ADDRESS]))


part = mock.part("val")
matches = decide(scored[isin(scored[C.S1_ID], pd.Index(part.s1[C.ENTITY_ID]))], rule)
counts = {k: len(error_samples(matches, part, k, n=10**9))
          for k in ("false_merge", "missed", "false_singleton", "singleton_merge")}
print("v104:", counts, "\nv103:", parent["metrics"]["errors_mock"]["mock rule"])
for kind in ("false_merge", "singleton_merge", "missed"):
    print(f"--- {kind}")
    display(with_raw(error_samples(matches, part, kind, n=10, scored=scored)))

v104: {'false_merge': 5265, 'missed': 68406, 'false_singleton': 3325, 'singleton_merge': 297} 
v103: {'false_merge': 6311, 'missed': 79354, 'false_singleton': 3534, 'singleton_merge': 445}
--- false_merge


,source1_entity_id,entity_id,prob,name_l,addr_l,name_r,addr_r
0,S1-107585328,S2-474103230,0.910530,Simba International Ltd,"New Delhi, B-4/13, 2Nd Floor, Main Wali Nagar,...",Simba International,
1,S1-137695892,S3-697508841,0.901650,Classic Suisse LLC,"Phoenix, 15801 48th Street, AZ, Unit 1216",Classic Suisse LLC,
2,S1-176450615,S3-332436853,0.952157,Gurgaon India Private Limited,"Gurgaon, Dlf Building No.9, Haryana, Tower-A, ...",Gurgaon India Private Ltd,"B-03, Gurgaon, Gurugram, HR"
3,S1-361975205,S2-766494919,0.918741,Bangalore South Systems Private Limited,"#004, Ashirwadh, 13Th Cross, Bangalore South, ...",Private Bangal0re South Systems Limited,"Karnataka, BANGALORE, BANGALORE SOUTH, NO. 110/1"
4,S1-377917849,S3-694776458,0.777118,Rks & Co,"1St Floor Room No 11 16/B Shakespeare Sarani, ...",Rks &-Co,
5,S1-489913517,S2-77179061,0.996783,Cornerstone Medicals Private Limited,"C/O Trimbak Gavade, House No-R. 30-02, Dhangar...",Cornerstone Mega Private Limited,"DOOR NO 99 C/O TRIMBAK GAVADE, HOUSE NO-R. 30-..."
6,S1-738128729,S2-959360977,0.988650,Faclara Capital Partners LLC,"8 Web Road, Georgetown, MA",Faclara Capital Partners,"29 WEB ROAD, MA, GEORGETOWN"
7,S1-835179861,S2-90631569,0.804262,Premier Global LLP,"505, Gf Nyay Khand-3, Indirapuram, Ghaziabad, ...",PREMIER SERVICES LLP,"509, GF NYAY KHAND-3, INDIRAPURAM, Uttar Pradesh"
8,S1-893959950,S3-746174645,0.938408,Vikram & Partners,"3/17 Azadgarh, Kolkata, Howrah, West Bengal",Vikram Vikram Partners,
9,S1-89401279,S3-692503117,0.923117,Technology Mnr India Pvt Ltd,"Transcon Triumph 704, Tower A Off New Link Roa...",Technology Mnr Infra Pvt Ltd,"Transcon Triumph 2-711, Mumbai, Mumbai City, MH"


--- singleton_merge


,source1_entity_id,entity_id,prob,name_l,addr_l,name_r,addr_r
0,S1-113976931,S3-868477138,0.922584,East Delhi Life Private Limited,"G 306 Ground Floor Old No, Ghazipur, East Delh...",Private EAST Delhi LIFE Limited,"5921-A, East Delhi, Delhi, DL"
1,S1-143794032,S3-497831350,0.987562,Electrum Clinic,"Ramesh Nilaya, 1St Cross, K R Extension, Tumku...",Electrum Projects,"Door No 45 Ramesh Nilaya, 1St Cross, K R Exten..."
2,S1-181116332,S2-765632443,0.964678,Smyrna Animal Hospital Inc.,"110 Creek Court, Smyrna, TN",Smyrna Animal Hospital Inc,"TN, SMYRNA, 114 CREEK CT"
3,S1-199537675,S3-208087199,0.897128,Mangalam Transformation Pvt. Ltd.,"Deepam Thumpaly, Irenipuram-629 197. Kanyakuma...",Mangalam Transformation Public Limited,"Door No 82 Deepam Thumpaly, Irenipuram-629 197..."
4,S1-30056058,S2-957804806,0.996696,Osprey Group,"231 Silvermine Avenue, Norwalk, CT",Osprey Group,"232 Silvermine Ave, NORWALK, CT"
5,S1-369692708,S2-534818635,0.975765,Aqube (India) Human South 24 Parganas,"South 24 Parganas, Tentul Baria, West Bengal, ...",Aqube (India) Technologies South 24,"পশ্চিমবঙ্গ, H.NO 27 TENTUL BARIA, 3RD FLOOR, M..."
6,S1-891416765,S3-668192809,0.721646,Exports Tri Fund Private Limited,Ground Nursing Ho Grandness B Wing Cts No 2408...,PRIVATE EXPORTS TRI FUND LIMITED,"No Rd Floor, Mumbai Ii, MH"
7,S1-913800724,S3-383543615,0.831491,Akar Tech Pvt Ltd,"No.641/2C (2), Sundamedu Near Venkateshwarawei...",akar tech ltd,
8,S1-941195609,S3-633263118,0.852620,Impex Farmer,"Sanatana Dharma Sabha Sector - 8, R. K. Puram,...",Impex Trade,"New Delhi, Sanatana Dharma Sabha Sector - 8, R..."
9,S1-989446686,S3-584395560,0.973535,Garrett Sterling Alpha L.L.C.,"2866 Rd 164, OH, Cardington",Garrett LLC Alpha Sterling,"2871 Rd 164, Cardington, Ohio"


--- missed


,source1_entity_id,entity_id,prob,name_l,addr_l,name_r,addr_r
0,S1-274554043,S2-843741800,NaN,Baba Food,"Office No-219, 11/5 Second Floor South Tukogan...",बाबा फूड,"277 OFICE NO-219, INDORE, NULL, Madhya Pradesh"
1,S1-555918400,S2-838047033,0.601856,Farmer Dental Associates,"Unit 1/2, Cambridge, 1104 Gaston Avenue, OH",FARMER DENTAL ASSOCIATES [CO],"CABRIDGE, 001104 GASTON AVENUE, OH"
2,S1-583575761,S3-696689157,NaN,Ranjit & Brothers Corporation,"Madhav Heritage, 1641 Sadashiv Peth, Tilak Roa...",Ranjit &,"Madhav Heritage, Pune, NULL, MH"
3,S1-585940696,S3-706624556,0.676117,Capital Alphabet,"25 Hunting Hollow Drive, Pepper Pike, OH",Calovera Labs,"25 Hunting Hollow Drive, Pepper Pike, OH"
4,S1-634305339,S2-131572980,NaN,Universal Exports Private Limited,"C-1, G-11, Ground Floor, Krishna Apra Plaza, S...",यूनिवर्सल एक्सपोर्ट्स प्राइवेट लिमिटेड,"C-##1, LUCKNOW HQ REGION, उत्तर प्रदेश"
5,S1-718326235,S3-740299382,NaN,HHD Beverages Private Limited,"1/25, Hazuri Bhawan, Peepal Mandi Road, Agra, ...",Ariapyra,"Agra, 1/25, UP, Agra"
6,S1-825888031,S3-345500730,NaN,Kolkata Service,"Howrah, West Bengal, Kolkata, Kaliprasanna Roy...",kolkataservice.com,"33, Howrah, Kolkata, WB"
7,S1-835179861,S2-817775873,NaN,Premier Global LLP,"505, Gf Nyay Khand-3, Indirapuram, Ghaziabad, ...",प्रीमियर ग्लोबल एलएलपी,"GF NYAY KHAND-3, Uttar Pradesh, PLOT 665 505, ..."
8,S1-857894194,S2-660202965,NaN,"Sephira R. Nunez, Esq.","Groton, MA, Unit F, 12 Brookfield Drive",sephirarnunez.com,"MA, 12A BROOKFIELD DR, GROTON"
9,S1-98315586,S2-315720146,0.696754,Delta Homecare,"1225 Osteen Street, Unit 1, Vidor, TX",Umbraectoorbi,"1225 OSTEEN ST, VIDOR, TX"


## 7. Log the result

`mock_f05` is the decision number (KEEP when it beats v103 by more than 0.002). `local_f05`
stays empty: stage 2 is trained for the test's density, and the plain val fold is not
that world. `cand_recall` is the candidate recall after the filter on the mock val entities.

In [10]:
ts = TwoStage(stage1, models, rule, tcfg, table,
              {"fit": fit_info, "filter_report": filter_report.to_dict("index")})
ts.save(ARTIFACTS)
record = {
    "hypothesis": "a stage-2 matcher with competition features, trained at test density, "
                  "beats v103 on the mock; the stage-1 filter cuts candidates to ~10 per S1",
    "two_stage": tcfg.record(), "rule": asdict(rule),
    "mock_f_beta": res.loc["all", "f_beta"],
    **{f"mock_{c}": res.loc[c, "f_beta"] for c in ("India", "US")},
    **{k: res.loc["all", k] for k in ("f_beta_singletons", "f_beta_matched",
                                      "pair_precision", "pair_recall")},
    "cand_recall_val": filter_report.loc["val", "pair_recall"],
    "cands_mean_val": filter_report.loc["val", "candidates_mean"],
    "cand_recall_val_before": before.loc["val", "pair_recall"],
    "cands_mean_val_before": before.loc["val", "candidates_mean"],
    "tune_f_beta": float(table["f_beta"].max()),
    "importance_top": importance.sort_values(ascending=False).head(25).to_dict(),
    "ablation": ablation.to_dict("index"),
    "errors_mock": counts, "fit_info": fit_info, **timings, "peak_rss_gb": peak_rss_gb(),
}
DECISION = "KEEP" if record["mock_f_beta"] > float(parent["mock_f05"]) + 0.002 else "DROP"
record["decision"] = DECISION
print("v103", parent["mock_f05"], "-> v104", round(record["mock_f_beta"], 4), DECISION)
row = log_result(
    EXP_DIR, change="two-stage: v101 stage 1 + top-16 filter; stage 2 (XGBoost GPU) with "
                    "competition + anchor features trained on the mock fit entities",
    group="C5", mock_f05=record["mock_f_beta"], cand_recall=record["cand_recall_val"],
    notes=(f"cands {record['cands_mean_val']:.1f}/S1 (was {record['cands_mean_val_before']:.1f}); "
           f"singletons {record['f_beta_singletons']:.4f}"),
    metrics=record, owner="M1", parent="v103", decision=DECISION)
row

v103 0.9704 -> v104 0.9744 KEEP


{'version': 'v104',
 'date': '2026-09-26',
 'group': 'C5',
 'change': 'two-stage: v101 stage 1 + top-16 filter; stage 2 (XGBoost GPU) with competition + anchor features trained on the mock fit entities',
 'local_f05': '',
 'mock_f05': '0.9744',
 'cand_recall': '0.9644',
 'public_f05': '',
 'commit': '2f84bc8',
 'notes': 'cands 4.6/S1 (was 34.5); singletons 0.9868',
 'owner': 'M1',
 'parent': 'v103',
 'decision': 'KEEP'}

## 8. Conclusion

* **Two-stage beats the single stage at test density:** mock F0.5 **0.9744** against v103's
  0.9704 (+0.0040; India 0.9700, US 0.9792), est_public 0.9654 with this plain-tuned rule
  (0.9657–0.9659 tuned for the tight mock: v107).
* **Candidate set 7.5× smaller:** the stage-1 filter keeps 4.6 candidates per S1 (34.5 before)
  for −0.0011 candidate recall; the test candidate file shrinks from 802 MB accordingly.
* **What stage 2 uses:** `pool_gap` (the pair's p1 minus the record's best rival) carries 72 %
  of the gain, p1 19 %. Ablations: competition features +0.0018, anchors +0.0002 (cheap, kept).
* **Where recall still goes** (true pairs of the mock val entities): 3.45 % never become
  candidates, 0.94 % are lost to a rival S1 in the 1-to-1, 1.96 % fall below the rule, 0.11 %
  are filtered. Next: bigger blocking at density (v105 → v106) and rival features.
* **Decision:** KEEP; its rule is re-tuned for the tight mock in v107 (the upload).


## 9. Test inference (shortlisted)

`run_test_two_stage`: per country, v101's blocking (cached), stage 1 with the filter and the
competition features, stage 2 (mean of both models), the rule; `candidate_pairs.tsv` holds
exactly the filtered pairs stage 2 scored. Then the sanity table and both validators; the
files are kept in `submissions/v104/`.

In [11]:
t0 = time.time()
match_path, cand_path, s1n_test, test_matches, test_summary = run_test_two_stage(
    cfg, ts, cache_dir=STAGE1_CACHE / "test")
print(f"run_test {time.time() - t0:.0f} s -> {match_path}, {cand_path}")
country_of = s1n_test.set_index(C.ENTITY_ID)[C.COUNTRY]
n_s1 = s1n_test.groupby(C.COUNTRY).size()
by = test_matches[C.S1_ID].map(country_of)
display(pd.DataFrame({
    "s1": n_s1,
    "cands_per_s1": test_summary["n_cands"].groupby(test_summary.index.map(country_of)).sum() / n_s1,
    "matched_share": test_matches.groupby(by)[C.S1_ID].nunique() / n_s1,
    "matches_per_s1": test_matches.groupby(by).size() / n_s1}))
dest = C.ROOT / "submissions" / "v104"
dest.mkdir(parents=True, exist_ok=True)
for p in (match_path, cand_path):
    shutil.copy2(p, dest / p.name)
print("copied to", dest)

run_test 1243 s -> /home/suryaguru/StudioProjects/aws/business_entity_resolution/output/matching_results.tsv, /home/suryaguru/StudioProjects/aws/business_entity_resolution/output/candidate_pairs.tsv


,s1,cands_per_s1,matched_share,matches_per_s1
France,259452,6.039637,0.943916,3.310628
India,809986,4.837877,0.936299,3.218663
US,663106,4.990585,0.940224,3.317073


copied to /home/suryaguru/StudioProjects/aws/business_entity_resolution/submissions/v104


Both validators on the exact files that will be uploaded.

In [12]:
out = subprocess.run([sys.executable, "-m", "entity_resolution.submission", "--output-dir",
                      str(C.OUTPUT), "--check-ids"], capture_output=True, text=True)
print(out.stdout[-2000:], out.stderr[-2000:])
out = subprocess.run([sys.executable, str(C.OFFICIAL_VALIDATOR), "--matching", str(match_path),
                      "--candidate", str(cand_path), "--test-dir", str(C.DATASET / "test")],
                     capture_output=True, text=True)
print(out.stdout[-3000:], out.stderr[-2000:])
print(f"notebook total {time.time() - t_start:.0f} s, peak RSS {peak_rss_gb()} GB")

PASS
 


ML Challenge 2026 — submission validator
  test dir: /home/suryaguru/StudioProjects/aws/business_entity_resolution/dataset/student_resource/dataset/test
  required S1 entities: 1732544
  matching_results.tsv: 1732544 rows (105786 empty, 1626758 non-empty).
  candidate_pairs.tsv: 1732544 rows (20777 empty, 1711767 non-empty).

PASS — no blocking issues found. Safe to submit.
 
notebook total 4553 s, peak RSS 9.37 GB
